## Setup

In [1]:
import requests
import pandas as pd
import pyspark
from io import BytesIO
from dotenv import load_dotenv
import os

In [7]:
load_dotenv()
JDBC_PATH = os.getenv("JDBC_PATH")
_DB_URL = os.getenv("DB_URL")
_DB_NAME = os.getenv("DB_NAME")
_DB_USER = os.getenv("DB_USER")
_DB_PASSWORD = os.getenv("DB_PASSWORD")

if JDBC_PATH is None or _DB_URL is None or _DB_NAME is None or _DB_USER is None or _DB_PASSWORD is None:
    raise EnvironmentError("Set .env vars!")
if not os.path.exists(JDBC_PATH) or not os.path.isfile(JDBC_PATH):
    raise FileNotFoundError("JDBC driver not found!")

DB_URL = f"jdbc:mysql://{_DB_URL}/{_DB_NAME}"
DB_PROPERTIES = {"user": _DB_USER, "password": _DB_PASSWORD, "driver": "com.mysql.jdbc.Driver"}

In [8]:
spark = pyspark.sql.SparkSession.builder.appName("ETL TCC").config("spark.jars", JDBC_PATH).getOrCreate()

## Extract

### Base bueiros

In [3]:
BUEIROS_URL = "https://www.der.sp.gov.br/WebSite/Arquivos/DadosAbertos/AtivosRodoviarios/Bueiros/Bueiros.xlsx"
res = requests.get(BUEIROS_URL)
res.raise_for_status()

In [14]:
content_wrapper = BytesIO(res.content)
pandas_df = pd.read_excel(content_wrapper)
df_bueiros_raw = spark.createDataFrame(pandas_df)

### OpenMeteo

### Alagamentos CGE

### Mocks

In [17]:
pandas_df = pd.read_csv("./mocks/distritos.csv")
df_mock_distritos = spark.createDataFrame(pandas_df)

## Transform

## Load

In [10]:
spark.read.jdbc(DB_URL, "chuva", properties=DB_PROPERTIES)

DataFrame[id: int, qtd: int, dt_hr_inicio: timestamp, dt_hr_fim: timestamp, distrito_id: int]